In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
# block size - length of sequence
# batch size - number of sequences
max_iters = 10000
# eval interval = 2500
learning_rate = 3e-3 # variable- trying to get best performance and quality
eval_iters = 250 # only prints out at every eval iteration
droput = 0.2 # drops out random neurons so overfitting does not happen

cuda


In [52]:
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = (len(chars))

['\n', ' ', '!', '"', '&', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [53]:
string_to_int = { ch:i for i, ch in enumerate(chars)}
int_to_string = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([75,  1,  1,  1, 27, 63, 66, 63, 68, 56, 73,  1, 49, 62, 52,  1, 68, 56,
        53,  1, 46, 57, 74, 49, 66, 52,  1, 57, 62,  1, 38, 74,  0,  0,  0,  1,
         1, 24,  1, 29, 49, 57, 68, 56, 54, 69, 60,  1, 41, 53, 51, 63, 66, 52,
         1, 63, 54,  1, 43, 56, 53, 57, 66,  1, 24, 61, 49, 74, 57, 62, 55,  1,
        24, 52, 70, 53, 62, 68, 69, 66, 53, 67,  0,  1,  1,  1,  1, 57, 62,  1,
        49, 62,  1, 44, 62, 52, 53, 66, 55, 66])


In [54]:
split = int(0.8*len(data))
train_data = data[:split]
val_data = data[split:]

# returns a batch of input-target sequences from either the training or validation set
def get_batch(split):
    data = train_data if split == 'train' else val_data
    rand_indices_of_batch = torch.randint(len(data) - block_size, (batch_size,))
    # print(rand_indices_of_batch)
    input_seq = torch.stack([data[i:i+block_size] for i in rand_indices_of_batch])
    target_seq = torch.stack([data[i+1:i+block_size+1] for i in rand_indices_of_batch])
    input_seq, target_seq = input_seq.to(device), target_seq.to(device)
    return input_seq, target_seq

input_seq, target_seq = get_batch('train')
print('inputs:')
# print(input_seq.shape)
print(input_seq)
print('targets:')
print(target_seq)

inputs:
tensor([[63, 60, 10,  0, 39, 66, 53, 67],
        [49, 57, 66, 10,  1,  1, 43, 56],
        [ 1, 63, 54,  0, 70, 49, 60, 69],
        [52,  1, 67, 60, 53, 53, 70, 53]], device='cuda:0')
targets:
tensor([[60, 10,  0, 39, 66, 53, 67, 53],
        [57, 66, 10,  1,  1, 43, 56, 53],
        [63, 54,  0, 70, 49, 60, 69, 53],
        [ 1, 67, 60, 53, 53, 70, 53, 67]], device='cuda:0')


In [55]:
@torch.no_grad()
def estimate_loss():
    out = {}
    # Puts the model into evaluation mode, which changes how certain layers are treated- turns off dropout (need the whole network)
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            train_data, val_data = get_batch(split)
            logits, loss = model(train_data, val_data)
            losses[k] = loss.item()
        out[split] = losses.mean()
    # Puts the model into training mode, which changes how certain layers are treated- allows the use of dropout
    model.train()
    return out

In [56]:
class BigramLanguageModel(nn.Module):
    # creating an embedding table to predict the next character (aa, ab, ac, ad)
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # maps input token IDs to their corresponding learned embeddings (logits)
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            Batch, Time, Channels = logits.shape
            # need to get shape because view expects view(N, C)
            logits = logits.view(Batch * Time, Channels)
            targets = targets.view(Batch * Time)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    # generates new tokens
    def generate(self, index, max_new_tokens):
        # index is (Batch, Time) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the prediction
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (Batch, Channels)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (Batch, Channels)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (Batch, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (Batch, Time + 1)
        return index

# creating the model
model = BigramLanguageModel(vocab_size)
cuda_model = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(cuda_model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


C&'"6zsiRJ.fhCN;FmpYye6 3jviB5sm"IGnFPj5u0:Ca.DRjz-!ufo
G.';M3sfIUvVL!F53z-xK 3mk
tM7Z:nefwHikC"lcG&ox40;WKZgEbakSKHK3g5RWxRgI0j﻿-j.DVTQkgvD'3l)1Y-"KCUGGeAkAHN4Mk&Jaa&:nWc7oauTJ.4ZNYh;gJtd30.fnJ.FH7aaGJ6-g1GOskM3KhYqiRWvuD6)6-hBP.cPCI﻿PtfV?kqwx88bC:cE)Yb﻿'RjQaz'"oT:Yq)FgM9vu5C﻿7PO wikCKSfmWjOK!G!l.x﻿'aG6tM"11GIj&:1-hZ87sC0O-l:s9)8LOBG0P6kaUN;OM.'R3tfD); ?qjjTuvMogCQBJZJ9Exwx4,MBAk1x;jnyU;1 Oiho(n8ssBoG&n)wLh0u36uTRT'TyL

k1mZL8oDw30.62UaK8oGjpl-h("5,o.')Q;:dHfhFAAh?.FbmkJNjR
wihFjIGJ.YZnz3D.BZHF


In [57]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

# iter - iteration
for iter in range(max_iters):
    if iter % eval_iters == 0:
        # evaluates the loss at every eval iter
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    # sample a batch of data
    input_batch, target_batch = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(input_batch, target_batch)
    # ensures they do not add over time- previous gradients do not affect the current one
    optimizer.zero_grad(set_to_none = True) # most efficient
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.741, val loss: 4.766
step: 250, train loss: 4.143, val loss: 4.185
step: 500, train loss: 3.684, val loss: 3.716
step: 750, train loss: 3.341, val loss: 3.387
step: 1000, train loss: 3.124, val loss: 3.157
step: 1250, train loss: 2.937, val loss: 2.988
step: 1500, train loss: 2.799, val loss: 2.867
step: 1750, train loss: 2.736, val loss: 2.773
step: 2000, train loss: 2.668, val loss: 2.702
step: 2250, train loss: 2.648, val loss: 2.674
step: 2500, train loss: 2.578, val loss: 2.631
step: 2750, train loss: 2.575, val loss: 2.607
step: 3000, train loss: 2.550, val loss: 2.609
step: 3250, train loss: 2.524, val loss: 2.566
step: 3500, train loss: 2.497, val loss: 2.513
step: 3750, train loss: 2.506, val loss: 2.555
step: 4000, train loss: 2.495, val loss: 2.554
step: 4250, train loss: 2.463, val loss: 2.508
step: 4500, train loss: 2.473, val loss: 2.521
step: 4750, train loss: 2.465, val loss: 2.515
step: 5000, train loss: 2.459, val loss: 2.515
step: 5250, train l

***Types of optimizers***

1. **Mean Squared Error (MSE)**: MSE is a common loss function used in regresssion problems, where the goal is to predict a continous output. It measures the average squared difference between the predicted and actual values, and is often used to train neural networks for regression tasks.
2. **Gradient Descent (GD)**: is an optimization algorithm used to minimize the loss function of a machine learning model. The loss function measures how well the model is able to predict the target variable based on the input features. The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function.
3. **Momentum**: Momentum is an extension of SGD that adds a "momentum" term to the parameter updates. This term helps smooth out the updates and allow the optimizer to continue moving in the right direction, even is the gradient changes direction or varies in magnitude. Momentum is particularly useful for training deep neural networks.
4. **RMSprop**: RMSprop is an optimization algorithm that uses a moving average of the squared gradient to adapt the learning rate of each parameter. This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.
5. **Adam**: Adam is a popular optimization algorithm that combines the ideas of momentum and RMSprop. It uses a moving average of both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.
6. **AdamW**: AdamW is a modification of the Adam optimizer that adds weight decay to the paramter updates. This helps to regularize the mdoel and can improve generalization performance. We will be using the AdamW optimizer as it best suits the properties of the model we will train.

There are more optimizers and details at torch.optim

In [58]:
# generating with trained data- still doesn't look great
context = torch.zeros((1,1), dtype = torch.long, device = device)
generated_chars = decode(cuda_model.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


"
rstizmemad amun't oritwingaminind s. th s vesof prndonk o wathe yrlan, an ditan oppainsked jone d be chor.

d:!" S, in we avereatthecrdng ngin fuasacapod tintanetopinoreriglorethe n athand

f d des
ave t to, iove?"
oa " heansen wheritos ma  if
sunofed te kngatade

b, utheadinde f pisin hre
"wheng wn wen w, arerorod  toofrde Cindd.
warothe theyo Dore kfo thand
Dokedof sthir s m k!abu'r t Dod turg Pad ftourke n ave h

asowite and torthid wigeag an e. cabory d be taprar s Wo t f th Jiloth the t h
